# final_test — the sealed one-shot evaluation

**Read this before running anything below.** This notebook is the *only* place in the entire project that reads the sealed `final_test` split. Every other script, notebook, and dataset loader in this repository deliberately refuses to. `scripts/run_final_test.py` (which this notebook only invokes — no logic is inlined here) will:

- **Refuse to run at all** unless you pass `--i-understand-this-is-irreversible` explicitly.
- **Refuse to run twice.** If `final_test_report.json` already exists at the output root, it stops immediately rather than overwriting it. There is no resume and no `--overwrite` flag for this one.

Model, calibration, and threshold must already be fully frozen before you run this. As of this writing that means: controlled RINE seed 42, no temperature scaling (T=1 — a fitted temperature on the clean set was degenerate, see `docs/planning/nextSteps.md`), threshold 0.5. If any of that changes, this notebook must not be run until it changes back or the whole point of a held-out set is lost.

Run cells top to bottom, once, in a single session.

In [ ]:
# 1. GPU and Drive.
import torch
assert torch.cuda.is_available(), 'This requires a GPU Colab server'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Refresh repository and dependencies.
from pathlib import Path
import json
import shutil
import subprocess
import sys

PROJECT_ROOT = Path('/content/cya-techjam26')
REPOSITORY_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'


def run_script(*args, cwd=PROJECT_ROOT):
    result = subprocess.run(list(args), cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(str(a) for a in args)} failed with code {result.returncode}")
    return result


if (PROJECT_ROOT / '.git').is_dir():
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.splitlines()
    assert not status, f'Unexpected local changes in the remote checkout: {status}'
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
run_script(sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt')
run_script(sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps')

In [ ]:
# 3. Stage the fixed-Q96 manifest and matched-clean images (this bundle
#    contains every split, including final_test; nothing here filters yet).
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
input_archive = DRIVE_ARTIFACT_ROOT / 'task2_stagea_bundle.tar.gz'
assert input_archive.is_file(), input_archive
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(input_archive, TASK2_ROOT)
manifest = TASK2_ROOT / 'fixed_q96_manifest.csv'
assert manifest.is_file(), manifest
print('Manifest ready:', manifest)

In [ ]:
# 4. Restore the frozen controlled-RINE seed-42 checkpoint (read-only input;
#    nothing here trains or mutates it).
CHECKPOINT_ROOT = ARTIFACT_ROOT / 'robustness' / 'train-controlled-rine' / 'seed_42'
DRIVE_CHECKPOINT_ROOT = DRIVE_ARTIFACT_ROOT / 'robustness' / 'train-controlled-rine' / 'seed_42'
assert DRIVE_CHECKPOINT_ROOT.is_dir(), f'Checkpoint not found on Drive: {DRIVE_CHECKPOINT_ROOT}'
shutil.copytree(DRIVE_CHECKPOINT_ROOT, CHECKPOINT_ROOT, dirs_exist_ok=True)
checkpoint = CHECKPOINT_ROOT / 'best_50_50.pt'
assert checkpoint.is_file(), checkpoint
print('Checkpoint ready:', checkpoint)

## Stop and confirm before continuing

The next cell reads `final_test`. Re-read the note at the top of this notebook. Once you run it, it cannot be undone or rerun — `scripts/run_final_test.py` will refuse a second attempt at the same output root. Only continue if the model, calibration, and threshold are genuinely frozen.

In [ ]:
# 5. The sealed evaluation. Runs exactly once. Publishes the report to Drive
#    only after it succeeds.
FINAL_TEST_OUTPUT_ROOT = ARTIFACT_ROOT / 'final_test'
DRIVE_FINAL_TEST_ROOT = DRIVE_ARTIFACT_ROOT / 'final_test'

run_script(
    sys.executable, 'scripts/run_final_test.py',
    '--manifest', str(manifest),
    '--checkpoint', str(checkpoint),
    '--output-root', str(FINAL_TEST_OUTPUT_ROOT),
    '--device', 'cuda',
    '--i-understand-this-is-irreversible',
)

DRIVE_FINAL_TEST_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(FINAL_TEST_OUTPUT_ROOT, DRIVE_FINAL_TEST_ROOT, dirs_exist_ok=True)
report = json.loads((FINAL_TEST_OUTPUT_ROOT / 'final_test_report.json').read_text())
print(json.dumps(report, indent=2))

## Reading the result

`report['metrics']` has overall accuracy, per-class accuracy, false-positive/negative rates, ECE, and the confusion matrix, computed with `src/cya_detector/evaluation/metrics.py`'s `binary_metrics` — the same function used everywhere else in the project, on `final_test` for the first and only time. Record this result in `docs/planning/nextSteps.md`; do not retrain, recalibrate, or re-run against `final_test` based on what it shows.